# **Notebook 6: Pipeline Yolo V11**

*This notebook is used for round 1, round 2, testing and predicting parking spaces using YOLO version 11s.*

This Jupyter notebook runs the pipeline to train, test and predict parking spaces using Yolo V11s.

The script was written for a project commissioned by the Municipality of Wageningen in relation to the course Remote Sensing and GIS integration (GRS60312).

The authors of the script are Polly Cheung, Pascal Dubbelman, Anthony Jansen, Iris Lagemaat, and Susanna van de Wetering.

*Last edited: 23/06/2026*

In [ ]:
# Connect with Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

First, install ultralytics in order to use pre-trained YOLO object detection models and install plug-ins

In [ ]:
!pip install ultralytics

In [ ]:
# Install packages
from pathlib import Path
import sys
from ultralytics import YOLO

**Step 1:** Define paths

In [4]:
# Point to project root.
base = Path('/content/gdrive/MyDrive/ParkingSpaceDetection')

# Directories training and validation split. Change folders according to which round you want to run (round1 or round2).
train_val_tif = base / 'data' / 'round1' / 'images' / 'train_val'
train_val_xml = base / 'data' / 'round1' / 'labels'/ 'train_val'
train_val_png = base / 'data' / 'round1'/ 'images' / 'train_val_png'
train_val_txt = base / 'data' / 'round1' / 'labels' / 'train_val_txt'

train_png = base / 'data' / 'round1' / 'images' / 'train'
val_png   = base / 'data' / 'round1' / 'images' / 'val'
train_txt = base / 'data' / 'round1' / 'labels' / 'train'
val_txt   = base / 'data' / 'round1' / 'labels' / 'val'

# Directories test split.
test_tif = base / 'data' / 'roundTest' / 'test_tiles'
test_xml = base / 'data' / 'roundTest' / 'test_labels'
test_png = base / 'data' / 'roundTest' / 'images' / 'test'
test_txt = base / 'data' / 'roundTest' / 'labels' / 'test'

# Directories prediction split.
predict_tif = base / 'data' / 'predict'
predict_png = base / 'data' / 'images' / 'predict'

geojson_output_path = base / 'results' / 'predictionsYoloV11.geojson'
yaml_path = base / 'Object_Detection_Models' / 'data.yaml'
yolo11_output = base / 'yolo11_output'


**Step 2:** Convert the `.tif` files to `.png`

In [ ]:
sys.path.append(str(base / 'Object_Detection_Models'))

from pythonHBB.tif_to_png import tif_to_png

# Convert .tif to .png. Change input according to which round you want to convert (round1, round2 or roundTest).
tif_to_png(test_tif, test_png)

**Step 3:** Convert `.xml` files to `.txt` files

In [ ]:
sys.path.append(str(base / 'Object_Detection_Models'))

from pythonHBB.xml_to_txt import xml_to_txt

# Convert .xml to .txt. Change input according to which round you want to convert (round1, round2 or roundTest).
xml_to_txt(test_xml, test_txt)

**Step 4:** Create training and validation data

In [ ]:
sys.path.append(str(base / 'Object_Detection_Models'))

from pythonHBB.train_val_data_split import create_dirs, split_data, move_pairs

# Create directories for training and validation data.
create_dirs(train_png, val_png, train_txt, val_txt)

# Split data into 80% train, 20% val.
train_imgs, val_imgs = split_data(train_val_png)

# Move training and validation data into directories.
move_pairs(train_png, train_val_txt, train_png, train_txt)
move_pairs(val_png,   train_val_txt, val_png,   val_txt)

**Step 5:** Train model

In [ ]:
sys.path.append(str(base / 'Object_Detection_Models'))

from pythonHBB.train_model import train_model

# Model YOLO 11s.
yolo = YOLO("yolo11s.pt")

# Train model.
train_model(yolo, yaml_path, yolo11_output )

**Step 6:** Validate model

In [ ]:
# Load best trained model.
model = YOLO('/content/gdrive/MyDrive/ParkingSpaceDetection/yolo11_output/train/weights/best.pt')

# Validate the loaded 'best.pt' model on the validation dataset
validation_results = model.val(data=yaml_path, split='val')

**Step 7:** Test model on 'unseen' data

In [ ]:
# Validate test split.
test_val = model.val(data=yaml_path, split='test')

**Step 8:** Let the model predict on neighbourhood

In [ ]:
# Run prediction.
predict_results = model.predict(source=predict_png, save=True, save_txt=True, save_conf=True, conf=0.3)

**Step 9:** Georeference the predicted bounding boxes.

In [ ]:
sys.path.append(str(base))

from python.georeference_HBB import georeference_HBB

# Create .geojson from the predicted horizontal boudning boxes.
georeference_HBB(predict_results, predict_tif, geojson_output_path)